# Runoff flowing through a Himalayan basin (map animation)

Run the cells **from top to bottom**. You only need to edit **Step 4 (Settings)**.

| Step | What it does |
|---|---|
| 1 | Installs the Python packages (once) |
| 2 | Saves the two animation scripts next to this notebook (just run it) |
| 3 | Uploads your files into a `data/` folder |
| 4 | **Settings: replace the placeholders with your file names** |
| 5 | Checks your files and shows a quick preview map |
| 6 | Makes the video and shows it here |
| 7 | (Optional) Time-series animation of the same table |

**Files you need**
* Basin boundary: shapefile / GeoJSON / GeoPackage
* River network lines: shapefile / GeoJSON / GeoPackage
* DEM: GeoTIFF (or other raster)
* Runoff table: Excel/CSV with `Years, Months, QALLDTS, STotDTS, RTotDTS, GTotDTS, BTotDTS`

⚠️ A **shapefile is several files**: upload `.shp`, `.shx`, `.dbf` **and** `.prj` together (or zip them into one `.zip`; Step 3 unzips it).

## Step 1: Install packages (run once)

In [ ]:
%pip install -q geopandas rasterio scipy matplotlib pandas openpyxl imageio-ffmpeg ipywidgets

## Step 2: Save the animation scripts (just run; no editing needed)

In [ ]:
%%writefile runoff_animation.py
#!/usr/bin/env python3
"""
Animate runoff components (snowmelt, glacier melt, rainfall-runoff, baseflow)
for a basin as a stacked-area time series that draws itself over time, with a
side panel showing each component's value and share on the current date.

Input: a CSV with one row per time step (daily, monthly, ...), e.g.

    date,snowmelt,glacier_melt,rainfall,baseflow,observed
    2015-01-01,1.2,0.0,0.4,8.1,10.3
    ...

`observed` (measured discharge) is optional; when present it is drawn as a
line over the stacked components for comparison.

Examples
--------
    # your model output
    python runoff_animation.py --csv my_basin.csv --basin "Beas at Bhuntar" \
        --units "m³/s" --out runoff.mp4

    # column names differ in your file
    python runoff_animation.py --csv out.csv --date-col Date \
        --cols "Snow=snowmelt,Glacier=glacier_melt,Rain=rainfall,Base=baseflow" \
        --observed-col Qobs --out runoff.gif

    # SPHY-style monthly output (Years, Months, QALLDTS, STotDTS, RTotDTS,
    # GTotDTS, BTotDTS) - detected automatically; .xlsx/.csv/.txt all work
    python runoff_animation.py --csv sphy_output.xlsx --basin "My basin" \
        --out runoff.mp4

    # try it without data (synthetic, clearly labelled as such)
    python runoff_animation.py --demo --out demo.gif
"""
import argparse
import sys

import matplotlib
if "ipykernel" not in sys.modules:  # keep notebook inline plots working
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.animation import FFMpegWriter, FuncAnimation, PillowWriter

# Component order = stacking order (bottom -> top) = legend order.
COMPONENTS = ["snowmelt", "glacier_melt", "rainfall", "baseflow"]
LABELS = {
    "snowmelt": "Snowmelt",
    "glacier_melt": "Glacier melt",
    "rainfall": "Rainfall runoff",
    "baseflow": "Baseflow",
}
# Colour-blind-checked categorical palette (fixed order). Change here if you
# prefer domain colours; keep adjacent bands clearly distinct.
COLORS = {
    "snowmelt": "#2a78d6",
    "glacier_melt": "#eb6834",
    "rainfall": "#1baf7a",
    "baseflow": "#eda100",
}
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3df"


def make_writer(out, fps, bitrate):
    """GIF via Pillow; MP4 via ffmpeg (system ffmpeg, else the imageio-ffmpeg
    pip package, which bundles one)."""
    if out.lower().endswith(".gif"):
        return PillowWriter(fps=fps)
    if not FFMpegWriter.isAvailable():
        try:
            import imageio_ffmpeg
            plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
        except ImportError:
            sys.exit("ffmpeg not found. Run `pip install imageio-ffmpeg`, "
                     "or save as .gif instead.")
    return FFMpegWriter(fps=fps, bitrate=bitrate)


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--csv", help="input file (.csv, .txt tab/comma, .xlsx) "
                                 "with date (or Years+Months) + component columns")
    p.add_argument("--sheet", default=0, help="Excel sheet name/index")
    p.add_argument("--demo", action="store_true",
                   help="use synthetic demo data instead of a CSV")
    p.add_argument("--date-col", default="date")
    p.add_argument("--cols", default="",
                   help="map CSV columns to components: "
                        "'CsvName=snowmelt,CsvName2=glacier_melt,...'")
    p.add_argument("--observed-col", default="observed",
                   help="optional observed discharge column (ignored if absent)")
    p.add_argument("--basin", default="Himalayan basin")
    p.add_argument("--units", default="m³/s")
    p.add_argument("--start", help="first date to plot, e.g. 2015-01-01")
    p.add_argument("--end", help="last date to plot")
    p.add_argument("--resample", default="",
                   help="pandas rule to aggregate first, e.g. 'W' or 'MS' (mean)")
    p.add_argument("--window", type=int, default=0,
                   help="show only the last N time steps (scrolling view); "
                        "0 = whole period, axis fixed")
    p.add_argument("--step", type=int, default=None,
                   help="time steps advanced per frame "
                        "(default: 1 for monthly data, 3 otherwise)")
    p.add_argument("--fps", type=int, default=None,
                   help="frames per second (default: 6 for monthly, 20 otherwise)")
    p.add_argument("--dpi", type=int, default=120)
    p.add_argument("--out", default="runoff_animation.mp4",
                   help=".mp4 (needs ffmpeg) or .gif")
    return p.parse_args(argv)


def demo_data():
    """Synthetic daily series with a rough Himalayan seasonality.
    NOT real data - for testing the animation only."""
    rng = np.random.default_rng(42)
    t = pd.date_range("2016-01-01", "2018-12-31", freq="D")
    doy = t.dayofyear.values

    def bump(center, width, amp):
        return amp * np.exp(-0.5 * ((doy - center) / width) ** 2)

    snow = bump(150, 30, 120) * rng.uniform(0.85, 1.15, len(t))      # pre-monsoon
    glacier = bump(215, 25, 90) * rng.uniform(0.9, 1.1, len(t))      # peak summer
    rain = bump(205, 28, 110) * rng.gamma(2.0, 0.5, len(t))          # monsoon, spiky
    base = 40 + bump(240, 60, 45) + rng.normal(0, 1.5, len(t))
    df = pd.DataFrame({"snowmelt": snow, "glacier_melt": glacier,
                       "rainfall": rain, "baseflow": base}, index=t)
    df["observed"] = df.sum(axis=1) * rng.normal(1.0, 0.07, len(t))
    return df.clip(lower=0)


# SPHY-style column names -> components (QALLDTS is the simulated total)
SPHY_COLS = {"STotDTS": "snowmelt", "GTotDTS": "glacier_melt",
             "RTotDTS": "rainfall", "BTotDTS": "baseflow", "QALLDTS": "total"}


def read_table(path, sheet):
    if path.lower().endswith((".xlsx", ".xls")):
        try:
            sheet = int(sheet)
        except ValueError:
            pass
        return pd.read_excel(path, sheet_name=sheet)
    # sep=None sniffs comma / tab / semicolon
    return pd.read_csv(path, sep=None, engine="python")


def years_months_to_date(df, ycol, mcol):
    """Build a date from a Years column (often filled only on January rows)
    and a Months column holding names ('January', 'June ') or numbers."""
    years = pd.to_numeric(df[ycol], errors="coerce").ffill()
    m = df[mcol].astype(str).str.strip()
    month_num = pd.to_numeric(m, errors="coerce")
    by_name = pd.to_datetime(m.str[:3], format="%b", errors="coerce").dt.month
    month_num = month_num.fillna(by_name)
    if years.isna().any() or month_num.isna().any():
        bad = df.loc[years.isna() | month_num.isna(), [ycol, mcol]]
        sys.exit(f"Could not parse year/month in rows:\n{bad.head()}")
    return pd.to_datetime(dict(year=years.astype(int),
                               month=month_num.astype(int), day=1))


def load_data(a):
    if a.demo:
        return demo_data()
    if not a.csv:
        sys.exit("Give --csv <file> or --demo")
    df = read_table(a.csv, a.sheet)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(how="all")
    lower = {c.lower(): c for c in df.columns}
    if a.date_col not in df.columns and "years" in lower and "months" in lower:
        df[a.date_col] = years_months_to_date(df, lower["years"], lower["months"])
    if not a.cols:
        sphy = {lower[k.lower()]: v for k, v in SPHY_COLS.items() if k.lower() in lower}
        if sphy:
            print(f"Using SPHY column mapping: {sphy}")
            df = df.rename(columns=sphy)
    if a.cols:
        mapping = dict(kv.split("=") for kv in a.cols.split(","))
        bad = set(mapping.values()) - set(COMPONENTS)
        if bad:
            sys.exit(f"--cols targets must be among {COMPONENTS}, got {sorted(bad)}")
        df = df.rename(columns=mapping)
    if a.date_col not in df.columns:
        sys.exit(f"Date column '{a.date_col}' not found. Columns: {list(df.columns)}")
    missing = [c for c in COMPONENTS if c not in df.columns]
    if missing:
        sys.exit(f"Missing component columns {missing}. Columns: {list(df.columns)}. "
                 "Use --cols to map them.")
    df[a.date_col] = pd.to_datetime(df[a.date_col])
    df[COMPONENTS] = df[COMPONENTS].apply(pd.to_numeric, errors="coerce")
    if "total" in df.columns:
        diff = (df[COMPONENTS].sum(axis=1) - pd.to_numeric(df["total"], errors="coerce")).abs()
        rel = diff / pd.to_numeric(df["total"], errors="coerce").abs().clip(lower=1e-9)
        n_bad = int(((diff > 1e-3) & (rel > 0.01)).sum())
        if n_bad:
            print(f"Warning: in {n_bad} rows the components do not sum to the "
                  "total column (>1% off). Check your columns.")
        else:
            print("Check OK: components sum to the total column.")
    keep = COMPONENTS + ([a.observed_col] if a.observed_col in df.columns else [])
    df = df.set_index(a.date_col)[keep].sort_index()
    if a.observed_col in df.columns:
        df = df.rename(columns={a.observed_col: "observed"})
    return df


def main(argv=None):
    """Run from Python/Jupyter with main(["--flag", "value", ...])."""
    a = parse_args(argv)
    df = load_data(a)
    if a.start or a.end:
        df = df.loc[a.start:a.end]
    if a.resample:
        df = df.resample(a.resample).mean()
    n_nan = int(df[COMPONENTS].isna().sum().sum())
    if n_nan:
        print(f"Warning: {n_nan} missing component values filled with 0 for plotting.")
    df[COMPONENTS] = df[COMPONENTS].fillna(0)
    if len(df) < 2:
        sys.exit("Need at least 2 time steps.")
    monthly = bool((df.index.day == 1).all() and
                   pd.Series(df.index).diff().dt.days.median() >= 28)
    if a.step is None:
        a.step = 1 if monthly else 3
    if a.fps is None:
        a.fps = 6 if monthly else 20
    date_fmt = "%b %Y" if monthly else "%d %b %Y"

    has_obs = "observed" in df.columns
    dates = df.index
    vals = df[COMPONENTS].to_numpy().T            # (4, n)
    cum = np.cumsum(vals, axis=0)                 # stacked tops
    total = cum[-1]
    ymax = max(total.max(), df["observed"].max() if has_obs else 0) * 1.08

    # ---- figure layout -------------------------------------------------
    plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRID,
                         "axes.labelcolor": INK_2, "xtick.color": INK_2,
                         "ytick.color": INK_2, "text.color": INK})
    fig = plt.figure(figsize=(12, 5.6), facecolor=SURFACE)
    gs = fig.add_gridspec(1, 2, width_ratios=[3.2, 1], wspace=0.32,
                          left=0.07, right=0.97, top=0.84, bottom=0.12)
    ax = fig.add_subplot(gs[0])
    axb = fig.add_subplot(gs[1])
    for x in (ax, axb):
        x.set_facecolor(SURFACE)
        for s in ("top", "right"):
            x.spines[s].set_visible(False)

    title = f"Runoff components — {a.basin}"
    if a.demo:
        title += "  (SYNTHETIC DEMO DATA)"
    fig.text(0.07, 0.94, title, fontsize=14, weight="bold", color=INK)
    date_txt = fig.text(0.07, 0.885, "", fontsize=11, color=INK_2)

    ax.set_ylim(0, ymax)
    ax.set_ylabel(f"Discharge ({a.units})")
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    if not a.window:
        ax.set_xlim(dates[0], dates[-1])

    # side panel: current value per component, horizontal bars
    # (bottom-to-top, same order as the stacked bands)
    ypos = np.arange(len(COMPONENTS))
    bars = axb.barh(ypos, np.zeros(len(COMPONENTS)), height=0.62,
                    color=[COLORS[c] for c in COMPONENTS])
    axb.set_yticks(ypos, [LABELS[c] for c in COMPONENTS])
    axb.set_xlim(0, vals.max() * 1.45)
    axb.tick_params(axis="y", length=0)
    axb.spines["left"].set_visible(False)
    axb.set_xlabel(f"Current ({a.units})")
    bar_txt = [axb.text(0, y, "", va="center", ha="left", fontsize=9, color=INK)
               for y in ypos]

    # legend (identity never colour-only: legend + labelled side panel)
    handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[c]) for c in COMPONENTS]
    names = [LABELS[c] for c in COMPONENTS]
    if has_obs:
        handles.append(plt.Line2D([], [], color=INK, lw=1.6))
        names.append("Observed")
    ax.legend(handles, names, loc="upper left", ncol=len(names), frameon=False,
              bbox_to_anchor=(0, 1.02), fontsize=9, handlelength=1.2)

    frames = list(range(1, len(dates), a.step))
    if frames[-1] != len(dates) - 1:
        frames.append(len(dates) - 1)
    hold = a.fps * 2                          # pause on the final frame
    frames += [len(dates) - 1] * hold

    dyn = []                                  # artists redrawn each frame

    def update(i):
        for art in dyn:
            art.remove()
        dyn.clear()
        lo = max(0, i - a.window) if a.window else 0
        sl = slice(lo, i + 1)
        x = dates[sl]
        prev = np.zeros(i + 1 - lo)
        for k, c in enumerate(COMPONENTS):
            top = cum[k, sl]
            # thin surface-coloured edge keeps adjacent bands separated
            dyn.append(ax.fill_between(x, prev, top, color=COLORS[c],
                                       lw=0.6, edgecolor=SURFACE))
            prev = top
        if has_obs:
            dyn.append(ax.plot(x, df["observed"].to_numpy()[sl], color=INK,
                               lw=1.4)[0])
        dyn.append(ax.axvline(dates[i], color=INK_2, lw=0.8, ls="--"))
        if a.window:
            ax.set_xlim(dates[lo], dates[max(i, lo + 1)])

        cur = vals[:, i]
        tot = cur.sum()
        for b, t, v, y in zip(bars, bar_txt, cur, ypos):
            b.set_width(v)
            share = 100 * v / tot if tot > 0 else 0
            t.set_position((v + axb.get_xlim()[1] * 0.02, y))
            t.set_text(f"{v:,.1f}  ({share:.0f}%)")
        obs = (f"   ·   observed {df['observed'].iloc[i]:,.1f} {a.units}"
               if has_obs and pd.notna(df["observed"].iloc[i]) else "")
        date_txt.set_text(f"{dates[i].strftime(date_fmt)}   ·   total simulated "
                          f"{tot:,.1f} {a.units}{obs}")
        return dyn

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / a.fps,
                         blit=False)
    writer = make_writer(a.out, a.fps, 2400)
    print(f"Rendering {len(frames)} frames -> {a.out}")
    anim.save(a.out, writer=writer, dpi=a.dpi, savefig_kwargs={"facecolor": SURFACE})
    plt.close(fig)
    print("Done.")
    return a.out


if __name__ == "__main__":
    main()

In [ ]:
%%writefile basin_flow_animation.py
#!/usr/bin/env python3
"""
Map animation of runoff flowing through a basin's river network.

Coloured particles (snowmelt, glacier melt, rainfall runoff, baseflow) are
released along the rivers and travel downstream to the outlet. The number of
particles released for each component follows your monthly runoff table, and
the rivers widen as total runoff rises.

Inputs
------
  --boundary  basin outline (shapefile / GeoJSON / GeoPackage)
  --rivers    river network lines (any digitising direction; flow direction is
              worked out from the network and the outlet)
  --dem       elevation raster (GeoTIFF, PCRaster .map, ...) - hillshade
              background, outlet detection, and where particles start
  --data      runoff table: SPHY-style Years/Months/QALLDTS/STotDTS/RTotDTS/
              GTotDTS/BTotDTS (.xlsx/.csv/.txt), or date + component columns
  --glaciers  (optional) glacier outlines, e.g. RGI; glacier-melt particles
              then start inside glaciers instead of the highest terrain

IMPORTANT - what is and is not data-driven
  * Amount of each component through time: from your table (outlet values).
  * WHERE particles start is illustrative: the table holds outlet totals, not
    per-cell values, so snowmelt starts on high ground, glacier melt on the
    highest ground (or in --glaciers polygons), rain and baseflow anywhere.
  * Particle speed is not to scale (real travel times are hours to days).
  The figure carries a footnote saying this.

Example
-------
    python basin_flow_animation.py --boundary basin.shp --rivers rivers.shp \\
        --dem dem.tif --data sphy_output.xlsx --basin "My basin" \\
        --start 2000-01 --end 2002-12 --out basin_flow.mp4
"""
import argparse
import sys
from types import SimpleNamespace

import matplotlib
if "ipykernel" not in sys.modules:  # keep notebook inline plots working
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection

import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.warp import Resampling, calculate_default_transform, reproject
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components, dijkstra
from scipy.spatial import cKDTree

from runoff_animation import COMPONENTS, LABELS, load_data, make_writer

# Particle colours, checked for colour-blind separation when all four are mixed
# together on the dark map background (white/cyan/blue/amber).
COLORS = {
    "snowmelt": "#e8e6df",
    "glacier_melt": "#3fc1d6",
    "rainfall": "#3987e5",
    "baseflow": "#c98500",
}
BG = "#1a1a19"
INK = "#ffffff"
INK_2 = "#c3c2b7"
INK_3 = "#8a8980"
RIVER = "#5b6f86"


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--boundary", required=True)
    p.add_argument("--rivers", required=True)
    p.add_argument("--dem", required=True)
    p.add_argument("--data", required=True, help="runoff table (see above)")
    p.add_argument("--sheet", default=0, help="Excel sheet name/index")
    p.add_argument("--glaciers", help="optional glacier polygons (e.g. RGI)")
    p.add_argument("--outlet", help="outlet as 'lon,lat' (WGS84). "
                                    "Default: lowest river end point on the DEM")
    p.add_argument("--basin", default="Himalayan basin")
    p.add_argument("--units", default="m³/s")
    p.add_argument("--start", help="first month, e.g. 2000-01")
    p.add_argument("--end", help="last month, e.g. 2002-12")
    p.add_argument("--frames-per-step", type=int, default=12,
                   help="animation frames per time step (month)")
    p.add_argument("--fps", type=int, default=24)
    p.add_argument("--travel-frames", type=int, default=60,
                   help="frames for a particle to travel the longest river path")
    p.add_argument("--max-spawn", type=float, default=45,
                   help="particles released per frame at the highest total runoff")
    p.add_argument("--snap", type=float, default=0,
                   help="snap distance (m) to join river lines that do not share "
                        "vertices exactly. Default: 2x the vertex spacing")
    p.add_argument("--spacing", type=float, default=0,
                   help="river vertex spacing in m (default: auto)")
    p.add_argument("--seed", type=int, default=1)
    p.add_argument("--dpi", type=int, default=110)
    p.add_argument("--out", default="basin_flow.mp4", help=".mp4 (ffmpeg) or .gif")
    return p.parse_args(argv)


# ---------------------------------------------------------------- GIS input
def load_dem(path, crs, max_px=1400):
    """Read the DEM reprojected to `crs`, at most `max_px` pixels wide."""
    with rasterio.open(path) as src:
        src_crs = src.crs
        if src_crs is None:
            sys.exit(f"{path} has no CRS. Assign one first (e.g. in QGIS).")
        transform, w, h = calculate_default_transform(
            src_crs, crs, src.width, src.height, *src.bounds)
        scale = max(1.0, w / max_px)
        w, h = int(w / scale), int(h / scale)
        transform = transform * transform.scale(scale, scale)
        dem = np.full((h, w), np.nan, dtype="float32")
        reproject(rasterio.band(src, 1), dem, src_nodata=src.nodata,
                  dst_transform=transform, dst_crs=crs, dst_nodata=np.nan,
                  resampling=Resampling.bilinear)
    return dem, transform


def sample(dem, transform, xy):
    col, row = ~transform * (xy[:, 0], xy[:, 1])
    row = np.clip(row.astype(int), 0, dem.shape[0] - 1)
    col = np.clip(col.astype(int), 0, dem.shape[1] - 1)
    return dem[row, col]


def hillshade(dem, transform, az=315, alt=45):
    z = np.where(np.isnan(dem), np.nanmin(dem), dem)
    dy, dx = np.gradient(z, abs(transform.e), transform.a)
    slope = np.pi / 2 - np.arctan(np.hypot(dx, dy))
    aspect = np.arctan2(-dx, dy)
    az, alt = np.radians(az), np.radians(alt)
    hs = np.sin(alt) * np.sin(slope) + np.cos(alt) * np.cos(slope) * np.cos(az - aspect)
    return np.clip(hs, 0, 1)


def build_network(rivers, spacing, snap):
    """River lines -> graph of evenly spaced vertices.
    Returns node coords, edge list (i, j, length)."""
    key_to_id, coords, node_line = {}, [], []
    edges = []
    q = spacing / 20  # vertices closer than this are merged

    def node(x, y, line_id):
        k = (round(x / q), round(y / q))
        if k not in key_to_id:
            key_to_id[k] = len(coords)
            coords.append((x, y))
            node_line.append(line_id)
        return key_to_id[k]

    ends = []
    for li, geom in enumerate(rivers.geometry):
        g = geom.segmentize(spacing)
        xy = np.asarray(g.coords)[:, :2]
        ids = [node(x, y, li) for x, y in xy]
        ends += [ids[0], ids[-1]]
        for a, b in zip(ids[:-1], ids[1:]):
            if a != b:
                edges.append((a, b, float(np.hypot(*np.subtract(coords[a], coords[b])))))
    coords = np.array(coords)
    node_line = np.array(node_line)

    # join line ends that touch another line without sharing a vertex
    tree = cKDTree(coords)
    deg = np.bincount(np.array([e[:2] for e in edges]).ravel(), minlength=len(coords))
    n_snapped = 0
    for e in set(ends):
        if deg[e] != 1:
            continue
        d, idx = tree.query(coords[e], k=min(12, len(coords)), distance_upper_bound=snap)
        for dist, j in zip(np.atleast_1d(d), np.atleast_1d(idx)):
            if np.isfinite(dist) and j != e and node_line[j] != node_line[e]:
                edges.append((e, int(j), max(float(dist), 1e-6)))
                n_snapped += 1
                break
    print(f"River network: {len(coords)} vertices, {len(edges)} edges, "
          f"{n_snapped} line ends snapped")
    return coords, np.array(edges)


def orient_to_outlet(coords, edges, outlet):
    """Shortest path along the network to the outlet -> downstream neighbour
    of every vertex (`down`) and distance to outlet (`dist`)."""
    n = len(coords)
    i, j, w = edges[:, 0].astype(int), edges[:, 1].astype(int), edges[:, 2]
    g = coo_matrix((np.r_[w, w], (np.r_[i, j], np.r_[j, i])), shape=(n, n)).tocsr()
    dist, pred = dijkstra(g, indices=outlet, return_predecessors=True)
    down = pred.astype(int)          # predecessor seen from the outlet = downstream
    down[outlet] = outlet
    return dist, down


# ---------------------------------------------------------------- main
def main(argv=None):
    """Run from Python/Jupyter with main(["--flag", "value", ...])."""
    a = parse_args(argv)
    rng = np.random.default_rng(a.seed)

    # runoff table (reuses the loader from runoff_animation.py)
    df = load_data(SimpleNamespace(demo=False, csv=a.data, sheet=a.sheet, cols="",
                                   date_col="date", observed_col="observed"))
    if a.start or a.end:
        df = df.loc[a.start:a.end]
    df[COMPONENTS] = df[COMPONENTS].fillna(0).clip(lower=0)
    if len(df) < 2:
        sys.exit("Need at least 2 time steps in the selected period.")
    monthly = bool((df.index.day == 1).all())
    q = df[COMPONENTS].to_numpy()                      # (T, 4)
    q_ref = q.sum(axis=1).max()

    # vector data in a metric CRS
    boundary = gpd.read_file(a.boundary)
    crs = boundary.estimate_utm_crs() if boundary.crs.is_geographic else boundary.crs
    boundary = boundary.to_crs(crs)
    basin_geom = boundary.union_all()
    rivers = gpd.read_file(a.rivers).to_crs(crs).explode(index_parts=False)
    rivers = rivers[rivers.geometry.notna() & (rivers.geom_type == "LineString")]
    if rivers.empty:
        sys.exit("No line features found in --rivers.")
    total_len = rivers.length.sum()
    spacing = a.spacing or max(total_len / 6000, 20)
    snap = a.snap or 2 * spacing

    dem, dtrans = load_dem(a.dem, crs)
    coords, edges = build_network(rivers, spacing, snap)
    z = sample(dem, dtrans, coords)

    # outlet
    n_comp, label = connected_components(coo_matrix(
        (np.ones(len(edges)), (edges[:, 0].astype(int), edges[:, 1].astype(int))),
        shape=(len(coords),) * 2), directed=False)
    if a.outlet:
        lon, lat = map(float, a.outlet.split(","))
        pt = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs=4326).to_crs(crs)[0]
        outlet = int(cKDTree(coords).query([pt.x, pt.y])[1])
    else:
        main_comp = np.bincount(label).argmax()
        cand = np.where(label == main_comp)[0]
        outlet = int(cand[np.nanargmin(np.where(np.isnan(z[cand]), np.inf, z[cand]))])
    print(f"Outlet at x={coords[outlet,0]:.0f}, y={coords[outlet,1]:.0f} ({crs}), "
          f"elevation {z[outlet]:.0f}")

    dist, down = orient_to_outlet(coords, edges, outlet)
    ok = np.isfinite(dist)
    if (~ok).any():
        print(f"Note: {(~ok).sum()} river vertices are not connected to the outlet "
              f"(drawn grey, no particles). Increase --snap if this looks wrong.")
    dmax = dist[ok].max()

    # upstream river length -> river width
    order = np.argsort(-np.where(ok, dist, -1))
    acc = np.zeros(len(coords))
    for v in order:
        if not ok[v] or v == outlet:
            continue
        acc[v] += dist[v] - dist[down[v]]
        acc[down[v]] += acc[v]
    seg_nodes = np.where(ok & (np.arange(len(coords)) != outlet))[0]
    segs = np.stack([coords[seg_nodes], coords[down[seg_nodes]]], axis=1)
    acc_frac = np.sqrt(acc[seg_nodes] / acc.max())

    # where each component's particles start (illustrative, see docstring)
    zr = pd.Series(z[ok]).rank(pct=True).to_numpy()   # elevation percentile
    nodes_ok = np.where(ok)[0]
    w_glacier = (zr >= 0.85).astype(float)
    if a.glaciers:
        gl = gpd.read_file(a.glaciers).to_crs(crs).union_all().buffer(spacing * 2)
        inside = gpd.GeoSeries(gpd.points_from_xy(*coords[nodes_ok].T), crs=crs).within(gl)
        if inside.any():
            w_glacier = inside.to_numpy().astype(float)
        else:
            print("Note: no river vertices inside --glaciers; using highest terrain.")
    weights = {
        "snowmelt": np.where(zr >= 0.4, zr ** 2, 0.0),
        "glacier_melt": w_glacier,
        "rainfall": np.ones_like(zr),
        "baseflow": np.ones_like(zr),
    }
    weights = {k: w / w.sum() for k, w in weights.items()}

    # ---- particle simulation (vectorised) -----------------------------------
    speed = dmax / a.travel_frames
    P = SimpleNamespace(node=np.empty(0, int), d=np.empty(0), c=np.empty(0, int))

    def rates_at(f):
        """Interpolated component runoff at animation frame f."""
        t = f / a.frames_per_step
        k = min(int(t), len(q) - 2)
        return q[k] + (q[k + 1] - q[k]) * min(t - k, 1.0)

    def step(rates):
        # release
        new_n, new_c = [], []
        for ci, comp in enumerate(COMPONENTS):
            n = rng.poisson(a.max_spawn * rates[ci] / q_ref)
            if n:
                new_n.append(rng.choice(nodes_ok, n, p=weights[comp]))
                new_c.append(np.full(n, ci))
        if new_n:
            nn = np.concatenate(new_n)
            P.node = np.r_[P.node, nn]
            P.d = np.r_[P.d, dist[nn]]
            P.c = np.r_[P.c, np.concatenate(new_c)]
        # move downstream
        P.d = P.d - speed * rng.uniform(0.85, 1.15, len(P.d))
        keep = P.d > 0
        P.node, P.d, P.c = P.node[keep], P.d[keep], P.c[keep]
        while True:
            m = dist[down[P.node]] > P.d
            if not m.any():
                break
            P.node[m] = down[P.node[m]]

    def positions():
        n0, n1 = P.node, down[P.node]
        span = np.maximum(dist[n0] - dist[n1], 1e-9)
        t = np.clip((dist[n0] - P.d) / span, 0, 1)[:, None]
        return coords[n0] * (1 - t) + coords[n1] * t

    # spin-up so the rivers are already flowing on the first frame
    for _ in range(a.travel_frames):
        step(q[0])

    # ---- figure --------------------------------------------------------------
    plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK_2,
                         "xtick.color": INK_3, "ytick.color": INK_3,
                         "axes.edgecolor": "#3a3a37"})
    fig = plt.figure(figsize=(12, 8.4), facecolor=BG)
    gs = fig.add_gridspec(2, 2, width_ratios=[3, 1], height_ratios=[4.2, 1],
                          left=0.03, right=0.97, top=0.9, bottom=0.1,
                          wspace=0.04, hspace=0.18)
    ax = fig.add_subplot(gs[0, 0])
    side = fig.add_subplot(gs[0, 1])
    tl = fig.add_subplot(gs[1, :])
    for x in (ax, side):
        x.set_facecolor(BG)
        x.set_axis_off()
    tl.set_facecolor(BG)

    # hillshade, masked to the basin
    hs = hillshade(dem, dtrans)
    mask = features.geometry_mask([basin_geom], dem.shape, dtrans)
    hs = np.ma.array(hs, mask=mask | np.isnan(dem))
    h, w = dem.shape
    ext = (dtrans.c, dtrans.c + w * dtrans.a, dtrans.f + h * dtrans.e, dtrans.f)
    ax.imshow(hs, cmap="gray", vmin=-0.4, vmax=2.6, extent=ext, interpolation="bilinear")
    boundary.boundary.plot(ax=ax, color=INK_2, lw=1.0)
    minx, miny, maxx, maxy = basin_geom.bounds
    pad = 0.03 * max(maxx - minx, maxy - miny)
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)
    ax.set_aspect("equal")

    if (~ok).any():
        bad = [(coords[int(i)], coords[int(j)]) for i, j, _ in edges
               if not (ok[int(i)] and ok[int(j)])]
        ax.add_collection(LineCollection(bad, colors="#44443f", linewidths=0.4))
    rivers_lc = LineCollection(segs, colors=RIVER, linewidths=0.5 + 2.5 * acc_frac,
                               capstyle="round", zorder=2)
    ax.add_collection(rivers_lc)
    trail = [ax.scatter([], [], s=s_, linewidths=0, alpha=al, zorder=3)
             for s_, al in ((5, 0.18), (7, 0.35))]   # previous frames = motion trail
    sc = ax.scatter([], [], s=10, linewidths=0, zorder=3)
    hist = []
    ax.plot(*coords[outlet], marker="v", ms=9, color=INK, mec=BG, zorder=4)
    ax.annotate("Outlet", coords[outlet], xytext=(8, -12), textcoords="offset points",
                color=INK_2, fontsize=9)
    pcolors = np.array([matplotlib.colors.to_rgba(COLORS[c]) for c in COMPONENTS])

    fig.text(0.03, 0.955, f"Runoff flowing through {a.basin}", fontsize=15,
             weight="bold")
    date_txt = fig.text(0.03, 0.918, "", fontsize=12, color=INK_2)
    fig.text(0.03, 0.02,
             "Amounts: monthly runoff components at the basin outlet (model output). "
             "Particle start locations are illustrative (by elevation"
             + (" / glacier outlines" if a.glaciers else "") +
             "), not modelled per cell.\nValues are interpolated between months for "
             "smooth motion. Particle speed is not to scale. "
             "One particle ≈ a fixed share of flow; river width scales with total "
             "runoff and upstream length.", fontsize=8, color=INK_3)

    # side panel: legend with live values (identity never colour-only)
    side.set_xlim(0, 1)
    side.set_ylim(0, 1)
    rows = []
    for k, comp in enumerate(COMPONENTS):
        y = 0.86 - k * 0.17
        side.scatter([0.06], [y], s=90, color=COLORS[comp])
        side.text(0.14, y + 0.025, LABELS[comp], fontsize=11, va="center")
        rows.append(side.text(0.14, y - 0.035, "", fontsize=10, color=INK_2,
                              va="center"))
    total_txt = side.text(0.06, 0.12, "", fontsize=11, color=INK)

    # timeline: stacked area of the whole period + cursor
    cum = np.cumsum(q, axis=1)
    prev = np.zeros(len(q))
    for ci, comp in enumerate(COMPONENTS):
        tl.fill_between(df.index, prev, cum[:, ci], color=COLORS[comp], alpha=0.85,
                        lw=0.4, edgecolor=BG)
        prev = cum[:, ci]
    tl.set_xlim(df.index[0], df.index[-1])
    tl.set_ylim(0, cum[:, -1].max() * 1.1)
    tl.set_ylabel(a.units, fontsize=9)
    for s in ("top", "right"):
        tl.spines[s].set_visible(False)
    tl.tick_params(labelsize=8)
    cursor = tl.axvline(df.index[0], color=INK, lw=1.2)

    n_frames = (len(q) - 1) * a.frames_per_step + 1
    date_fmt = "%B %Y" if monthly else "%d %b %Y"
    span = df.index[-1] - df.index[0]

    def update(f):
        r = rates_at(f)
        step(r)
        xy = positions()
        cols = pcolors[P.c] if len(P.c) else np.empty((0, 4))
        xy = xy if len(xy) else np.empty((0, 2))
        hist.append((xy, cols))
        del hist[:-3]
        for art, (hxy, hc) in zip(trail, hist[:-1][-2:]):
            art.set_offsets(hxy)
            art.set_color(hc)
        sc.set_offsets(xy)
        sc.set_color(cols)
        tot = r.sum()
        rivers_lc.set_linewidths((0.4 + 3.2 * acc_frac) * (0.35 + 0.65 * np.sqrt(tot / q_ref)))
        k = min(int(round(f / a.frames_per_step)), len(q) - 1)
        date_txt.set_text(df.index[k].strftime(date_fmt))
        for txt, v in zip(rows, r):
            share = 100 * v / tot if tot > 0 else 0
            txt.set_text(f"{v:,.1f} {a.units}  ·  {share:.0f}%")
        total_txt.set_text(f"Total  {tot:,.1f} {a.units}")
        cursor.set_xdata([df.index[0] + span * f / (n_frames - 1)] * 2)
        return sc, rivers_lc

    hold = a.fps
    frames = list(range(n_frames)) + [n_frames - 1] * hold
    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / a.fps, blit=False)
    writer = make_writer(a.out, a.fps, 4000)
    print(f"Rendering {len(frames)} frames -> {a.out}")
    anim.save(a.out, writer=writer, dpi=a.dpi, savefig_kwargs={"facecolor": BG})
    plt.close(fig)
    print("Done.")
    return a.out


if __name__ == "__main__":
    main()

## Step 3: Upload your files

* **Google Colab:** run the cell and click *Choose files*. Select all of your files (all shapefile parts, DEM, Excel). They are saved into `data/`.
* **Jupyter on your own computer:** either use the upload button the cell shows, **or** simply copy your files into the `data` folder next to this notebook (the cell creates it) and re-run the cell.

The cell finishes by listing what is in `data/`. Use those names in Step 4.

In [ ]:
import os, glob, zipfile

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def unzip_and_list():
    for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_DIR)
        print("Unzipped", z)
    print("\nFiles in data/:")
    for f in sorted(glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)):
        if os.path.isfile(f):
            print("  ", f.replace("\\", "/"))

try:
    from google.colab import files          # running in Google Colab
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(DATA_DIR, name), "wb") as fh:
            fh.write(content)
        if os.path.exists(name):
            os.remove(name)                  # Colab also saves a copy in the current folder
    unzip_and_list()
except ImportError:                          # local Jupyter
    try:
        import ipywidgets as widgets
        from IPython.display import display
        up = widgets.FileUpload(multiple=True, description="Choose files")
        btn = widgets.Button(description="Save to data/", button_style="primary")
        log = widgets.Output()

        def save(_):
            with log:
                val = up.value
                items = val.values() if isinstance(val, dict) else val  # ipywidgets 7 vs 8
                for it in items:
                    name = it["name"] if "name" in it else it["metadata"]["name"]
                    with open(os.path.join(DATA_DIR, name), "wb") as fh:
                        fh.write(bytes(it["content"]))
                    print("Saved", name)
                unzip_and_list()

        btn.on_click(save)
        print("Pick your files, then press 'Save to data/'. "
              "(Or copy them into the data folder by hand and re-run this cell.)")
        display(widgets.HBox([up, btn]), log)
    except ImportError:
        print("Copy your files into the 'data' folder next to this notebook, then re-run this cell.")
    unzip_and_list()

## Step 4: Settings ✏️ **edit this cell**

Replace every `YOUR_...` placeholder with the real file names listed at the end of Step 3.
Leave optional items as `None` if you don't have them.

In [ ]:
# ===================== EDIT HERE (placeholders) =====================
BOUNDARY_FILE = "data/YOUR_BASIN_BOUNDARY.shp"      # basin outline (.shp / .geojson / .gpkg)
RIVERS_FILE   = "data/YOUR_RIVER_NETWORK.shp"       # river lines
DEM_FILE      = "data/YOUR_DEM.tif"                 # elevation raster
RUNOFF_TABLE  = "data/YOUR_SPHY_RUNOFF_TABLE.xlsx"  # Years, Months, QALLDTS, STotDTS, ...
EXCEL_SHEET   = 0             # sheet number (0 = first) or sheet name, e.g. "Sheet1"

BASIN_NAME    = "YOUR BASIN NAME"
UNITS         = "m³/s"        # units of the runoff table; please check
START_MONTH   = None          # e.g. "1993-01"  (None = from first row)
END_MONTH     = None          # e.g. "1995-12"  (None = to last row)

GLACIER_FILE  = None          # optional, e.g. "data/RGI_glaciers.shp"
OUTLET_LONLAT = None          # optional, e.g. "77.10,31.95" (lon,lat). None = lowest river point

OUTPUT_VIDEO  = "basin_flow.mp4"   # or "basin_flow.gif"
FRAMES_PER_MONTH = 12         # more = slower, smoother
FPS = 24
# ====================================================================

## Step 5: Check files and preview (fix any ❌ before continuing)

In [ ]:
import os, importlib
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from types import SimpleNamespace
import runoff_animation; importlib.reload(runoff_animation)

problems = []
paths = {"BOUNDARY_FILE": BOUNDARY_FILE, "RIVERS_FILE": RIVERS_FILE,
         "DEM_FILE": DEM_FILE, "RUNOFF_TABLE": RUNOFF_TABLE, "GLACIER_FILE": GLACIER_FILE}
for k, p in paths.items():
    if p is None:
        continue
    if "YOUR_" in p:
        problems.append(f"❌ {k} is still a placeholder: {p}")
    elif not os.path.exists(p):
        problems.append(f"❌ {k}: file not found: {p}")
    elif p.lower().endswith(".shp"):
        base = os.path.splitext(p)[0]
        for ext in (".shx", ".dbf", ".prj"):
            if not (os.path.exists(base + ext) or os.path.exists(base + ext.upper())):
                problems.append(f"❌ {k}: missing {os.path.basename(base + ext)} "
                                "(upload all parts of the shapefile)")
if problems:
    print("\n".join(problems))
    raise SystemExit("Fix the problems above in Step 3 / Step 4, then re-run this cell.")

b = gpd.read_file(BOUNDARY_FILE); r = gpd.read_file(RIVERS_FILE)
with rasterio.open(DEM_FILE) as d:
    print(f"✅ DEM: {d.width}x{d.height} px, CRS {d.crs}")
print(f"✅ Boundary: {len(b)} feature(s), CRS {b.crs}")
print(f"✅ Rivers: {len(r)} line(s), CRS {r.crs}")
df = runoff_animation.load_data(SimpleNamespace(demo=False, csv=RUNOFF_TABLE, sheet=EXCEL_SHEET,
                                                cols="", date_col="date", observed_col="observed"))
print(f"✅ Runoff table: {len(df)} rows, {df.index[0]:%b %Y} to {df.index[-1]:%b %Y}")
display(df.head(13))

fig, ax = plt.subplots(figsize=(7, 7))
b.to_crs(r.crs).boundary.plot(ax=ax, color="k", lw=1)
r.plot(ax=ax, color="tab:blue", lw=0.8)
ax.set_title("Preview: boundary + rivers (check they line up)")
plt.show()

## Step 6: Make the animation (may take a few minutes)

In [ ]:
import importlib
import basin_flow_animation; importlib.reload(basin_flow_animation)
from IPython.display import Video, display

args = ["--boundary", BOUNDARY_FILE, "--rivers", RIVERS_FILE, "--dem", DEM_FILE,
        "--data", RUNOFF_TABLE, "--sheet", str(EXCEL_SHEET),
        "--basin", BASIN_NAME, "--units", UNITS, "--out", OUTPUT_VIDEO,
        "--frames-per-step", str(FRAMES_PER_MONTH), "--fps", str(FPS)]
if START_MONTH:   args += ["--start", START_MONTH]
if END_MONTH:     args += ["--end", END_MONTH]
if GLACIER_FILE:  args += ["--glaciers", GLACIER_FILE]
if OUTLET_LONLAT: args += ["--outlet", OUTLET_LONLAT]

out = basin_flow_animation.main(args)

if out.lower().endswith(".gif"):
    from IPython.display import Image
    display(Image(filename=out))
else:
    display(Video(out, embed=True, width=900))
try:
    from google.colab import files
    files.download(out)                 # Colab: also download the video
except ImportError:
    print("Video saved next to this notebook:", os.path.abspath(out))

### Notes for presenting
* **Amounts** of each component per month come from your table (basin-outlet values).
* **Where particles start is illustrative** (snowmelt on high ground, glacier melt on the highest ground or in `GLACIER_FILE`, rain and baseflow everywhere). The table has no per-cell values.
* Values are interpolated between months, and particle speed is not to scale. The video footnote says this.
* If it says river parts are *not connected to the outlet*, add `"--snap", "500"` (metres) to `args` in Step 6.
* If the outlet is in the wrong place, set `OUTLET_LONLAT` in Step 4.

## Step 7 (optional): Time-series animation of the same table

In [ ]:
import runoff_animation; importlib.reload(runoff_animation)
args2 = ["--csv", RUNOFF_TABLE, "--sheet", str(EXCEL_SHEET), "--basin", BASIN_NAME,
         "--units", UNITS, "--out", "runoff_timeseries.mp4"]
if START_MONTH: args2 += ["--start", START_MONTH]
if END_MONTH:   args2 += ["--end", END_MONTH]
out2 = runoff_animation.main(args2)
display(Video(out2, embed=True, width=900))